# Tip 영상 토픽 분류기 — KLUE-RoBERTa 멀티라벨 (딥러닝 실습)

Dewy `tipClassify.ts`(정규식 규칙)를 학습 신경망으로 교체. **멀티라벨** 텍스트 분류.

흐름: 데이터 로드 → baseline(규칙·TF-IDF) → KLUE-RoBERTa 학습 → gold 셋에서 비교.

> Colab: 런타임 → 런타임 유형 변경 → **GPU**. `data/tip_videos.jsonl`(필수),
> `data/gold.csv`(권장)를 업로드하거나 드라이브 마운트 후 `DATA_DIR` 를 맞춘다.


## 1. 의존성 & 설정


In [ ]:
# Colab 에서만 필요(로컬에 이미 깔려 있으면 skip)
!pip -q install transformers datasets scikit-learn accelerate 2>/dev/null


In [ ]:
import json, os, math, random
from pathlib import Path
import numpy as np, pandas as pd
import torch
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# tipClassify.ts TOPIC_PATTERNS 의 16개 라벨(순서 = 클래스 인덱스).
TOPICS = [
    'family_meeting','newlywed_home','wedding_gifts','legal_paperwork',
    'bridal_care','ceremony','wedding_hall','studio','dress_shop',
    'makeup_shop','hanbok','tailor_shop','honeymoon','appliance',
    'invitation_venue','general',
]
TOPIC2IDX = {t:i for i,t in enumerate(TOPICS)}
NUM_LABELS = len(TOPICS)

DATA_DIR = Path('data')        # Colab 에 업로드했다면 Path('.') 또는 드라이브 경로
MODEL_NAME = 'klue/roberta-base'   # 가볍게: 'klue/bert-base'
OUT_DIR = Path('model')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| labels:', NUM_LABELS)


## 2. 데이터 로드 (약한 라벨 = 규칙 출력)


In [ ]:
def to_multihot(label_lists):
    Y = np.zeros((len(label_lists), NUM_LABELS), dtype=np.float32)
    for i, labs in enumerate(label_lists):
        for l in labs:
            if l in TOPIC2IDX:
                Y[i, TOPIC2IDX[l]] = 1.0
    return Y

def parse_labels(cell):
    if isinstance(cell, list): return [c for c in cell if c in TOPIC2IDX]
    if not isinstance(cell, str) or not cell.strip(): return []
    return [c.strip() for c in cell.split('|') if c.strip() in TOPIC2IDX]

rows = [json.loads(l) for l in (DATA_DIR/'tip_videos.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
df = pd.DataFrame(rows)
df['labels'] = df['labels'].apply(parse_labels)
print('전체:', len(df))
print('라벨 0개(off-topic 추정):', (df['labels'].str.len()==0).sum())
print('평균 라벨 수:', round(df['labels'].str.len().mean(), 3))
df[['text','labels','search_query']].head(3)


## 3. 분할 — gold 우선 (weak supervision)

`data/gold.csv`(사람 손라벨)가 있으면 **test = gold**, 학습은 나머지 약한 라벨.
없으면 약한 라벨 holdout 으로 폴백하되, baseline 이 trivially 완벽해지므로
**'규칙 재현 측정일 뿐 실제 정확도 아님'** 을 경고한다(정직한 한계).


In [ ]:
gold_path = DATA_DIR/'gold.csv'
HAS_GOLD = gold_path.exists()

if HAS_GOLD:
    g = pd.read_csv(gold_path)
    g['y_true'] = g['label'].apply(parse_labels)
    g['y_rule'] = g['rule_labels'].apply(parse_labels) if 'rule_labels' in g else g['y_true']
    gold_ids = set(g['id'].astype(str))
    train_pool = df[~df['id'].astype(str).isin(gold_ids)].reset_index(drop=True)
    test_df = g.rename(columns={'y_true':'labels'})[['text','labels','y_rule']].reset_index(drop=True)
    print(f'[gold] test={len(test_df)} (손라벨), train_pool={len(train_pool)}')
else:
    print('[WARN] data/gold.csv 없음 → 약한 라벨 holdout 폴백.')
    print('       baseline(규칙)이 자기 라벨로 평가돼 F1≈1.0 — 실제 정확도 아님!')
    train_pool = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    cut = int(len(train_pool)*0.85)
    test_df = train_pool.iloc[cut:].copy(); test_df['y_rule'] = test_df['labels']
    train_pool = train_pool.iloc[:cut].reset_index(drop=True)

# train/val split (약한 라벨)
tp = train_pool.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
vcut = int(len(tp)*0.9)
train_df, val_df = tp.iloc[:vcut].reset_index(drop=True), tp.iloc[vcut:].reset_index(drop=True)
print('train', len(train_df), '| val', len(val_df), '| test', len(test_df))


## 4. Baseline — 규칙(정규식) & TF-IDF+LogReg


In [ ]:
def report(name, y_true, y_pred):
    micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f'{name:<28} micro-F1={micro:.3f}  macro-F1={macro:.3f}')
    return micro, macro

Y_test = to_multihot(test_df['labels'])

# (a) 규칙 baseline = 앱이 이미 저장한 정규식 출력(test_df.y_rule)
Y_rule = to_multihot(test_df['y_rule'])
results = {}
results['rule_regex'] = report('규칙(정규식, 앱 현재)', Y_test, Y_rule)
if not HAS_GOLD:
    print('   ↑ gold 없음 → 자기라벨 평가라 무의미. gold.csv 채우면 진짜 비교됨.')

# (b) 얕은 ML 비교군: TF-IDF + OneVsRest LogReg
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
vec = TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=2)
Xtr = vec.fit_transform(train_df['text']); Xte = vec.transform(test_df['text'])
Ytr = to_multihot(train_df['labels'])
clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=4.0))
clf.fit(Xtr, Ytr)
results['tfidf_logreg'] = report('TF-IDF + LogReg', Y_test, clf.predict(Xte))


## 5. 토크나이즈 & Dataset


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256

def make_ds(frame):
    d = Dataset.from_dict({'text': list(frame['text']),
                           'labels': to_multihot(frame['labels']).tolist()})
    return d.map(lambda b: tok(b['text'], truncation=True, max_length=MAX_LEN), batched=True)

ds_train, ds_val, ds_test = make_ds(train_df), make_ds(val_df), make_ds(test_df)
ds_train


## 6. 모델 & Trainer (multi-label)


In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, problem_type='multi_label_classification',
    id2label={i:t for i,t in enumerate(TOPICS)}, label2id=TOPIC2IDX)

THRESH = 0.5
def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    probs = 1/(1+np.exp(-logits))
    pred = (probs >= THRESH).astype(int)
    y = p.label_ids.astype(int)
    return {'micro_f1': f1_score(y, pred, average='micro', zero_division=0),
            'macro_f1': f1_score(y, pred, average='macro', zero_division=0)}

args = TrainingArguments(
    output_dir='ckpt', num_train_epochs=4, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='micro_f1', weight_decay=0.01, warmup_ratio=0.1,
    logging_steps=50, report_to='none', seed=SEED)

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  tokenizer=tok, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=compute_metrics)


## 7. 학습


In [ ]:
trainer.train()


## 8. 평가 — 모델 vs baseline (threshold sweep + per-class)


In [ ]:
logits = trainer.predict(ds_test).predictions
logits = logits[0] if isinstance(logits, tuple) else logits
probs = 1/(1+np.exp(-logits))

# threshold sweep — 멀티라벨은 임계값 튜닝이 중요
print('threshold sweep (test):')
best_t, best_f1 = 0.5, -1
for t in [0.3,0.35,0.4,0.45,0.5,0.55,0.6]:
    f = f1_score(Y_test, (probs>=t).astype(int), average='micro', zero_division=0)
    print(f'  t={t:.2f}  micro-F1={f:.3f}'); 
    if f>best_f1: best_t,best_f1=t,f
print('best threshold:', best_t)

pred = (probs>=best_t).astype(int)
results['klue_roberta'] = report('KLUE-RoBERTa (본 실습)', Y_test, pred)

print('\n=== 최종 비교표 ===')
print(f"{'method':<24}{'micro-F1':>10}{'macro-F1':>10}")
for k,(mi,ma) in results.items():
    print(f'{k:<24}{mi:>10.3f}{ma:>10.3f}')

print('\n=== per-class (모델) ===')
print(classification_report(Y_test, pred, target_names=TOPICS, zero_division=0))


## 9. 오분류 분석 — 규칙은 틀렸는데 모델이 맞춘 예


In [ ]:
# gold 가 있을 때 의미 있음(test_df.y_rule = 규칙, labels = 사람 정답)
wins = []
for i in range(len(test_df)):
    true = set(test_df['labels'].iloc[i]); rule = set(test_df['y_rule'].iloc[i])
    mdl = {TOPICS[j] for j in range(NUM_LABELS) if pred[i,j]}
    if mdl==true and rule!=true:
        wins.append((test_df['text'].iloc[i][:80], sorted(true), sorted(rule), sorted(mdl)))
print(f'규칙 오답 → 모델 정답: {len(wins)}건 (상위 5)')
for t,tr,ru,md_ in wins[:5]:
    print(f'\n· {t}\n   정답={tr}\n   규칙={ru}\n   모델={md_}')


## 10. 추론 데모 & 저장


In [ ]:
def predict(text, threshold=None):
    threshold = threshold or best_t
    enc = tok(text, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(model.device)
    with torch.no_grad():
        p = torch.sigmoid(model(**enc).logits)[0].cpu().numpy()
    return sorted([(TOPICS[i], float(p[i])) for i in range(NUM_LABELS) if p[i]>=threshold],
                  key=lambda x:-x[1])

print(predict('강남 호텔 웨딩홀 음식 시연 후기 + 본식 스냅 비용'))
print(predict('예비신부 다이어트 2주 식단 브이로그'))

OUT_DIR.mkdir(exist_ok=True)
trainer.save_model(str(OUT_DIR)); tok.save_pretrained(str(OUT_DIR))
print('saved ->', OUT_DIR.resolve())
